# Sinusoidal Positional Encoding — From First PrinciplesThis notebook walks through:1. **Why** transformers need positional encoding2. **How** the sinusoidal encoding creates a unique fingerprint per position3. **Visualizing** the frequency spectrum and the "clock" analogy4. **How** positional information flows into Q, K, V vectors and affects attention5. **Why** this approach struggles to scale beyond training length

In [ ]:
import torchimport torch.nn as nnimport matplotlib.pyplot as pltimport numpy as npimport seaborn as snssns.set_theme(style='whitegrid')%matplotlib inline

## Part 1: The Problem — Attention is Permutation InvariantSelf-attention computes:$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$If we shuffle the input tokens, Q, K, V all get shuffled the same way, and the outputis just a shuffled version of the original output. The model **cannot distinguish order**.Let's prove this:

In [ ]:
torch.manual_seed(42)d_model = 8seq_len = 4# Fake token embeddings (no positional info)X = torch.randn(1, seq_len, d_model)W_q = torch.randn(d_model, d_model)W_k = torch.randn(d_model, d_model)W_v = torch.randn(d_model, d_model)def attention(X, W_q, W_k, W_v):    Q = X @ W_q    K = X @ W_k    V = X @ W_v    scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)    weights = torch.softmax(scores, dim=-1)    return weights @ Vout_original = attention(X, W_q, W_k, W_v)# Shuffle: swap positions 1 and 2perm = [0, 2, 1, 3]X_shuffled = X[:, perm, :]out_shuffled = attention(X_shuffled, W_q, W_k, W_v)# Un-shuffle the outputinv_perm = [0, 2, 1, 3]out_unshuffled = out_shuffled[:, inv_perm, :]print('Max difference after un-shuffling:', (out_original - out_unshuffled).abs().max().item())print('\n→ Outputs are identical (up to float precision).')print('  Attention cannot tell "dog bites man" from "man bites dog".')

## Part 2: The Sinusoidal Positional Encoding FormulaFrom "Attention Is All You Need" (Vaswani et al., 2017):$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$Where:- `pos` = position in the sequence (0, 1, 2, ...)- `i` = dimension index (0, 1, 2, ..., d_model/2 - 1)- `d_model` = embedding dimensionEach pair of dimensions `(2i, 2i+1)` forms a **sin/cos pair** at a specific frequency.### Intuition: Think of it like a clock- Dimension pair 0: **seconds hand** (fast oscillation, high frequency)- Dimension pair 1: **minutes hand** (slower)- ...- Last dimension pair: **year hand** (very slow oscillation, low frequency)Just like a timestamp `14:32:07` uniquely identifies a time, the combination of allthese oscillations at different frequencies creates a **unique fingerprint** for each position.

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):    """    Generate the sinusoidal positional encoding matrix.    Returns: (max_len, d_model) tensor    """    pe = torch.zeros(max_len, d_model)    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)        # Each dimension pair oscillates at a different frequency.    # div_term = 1 / 10000^(2i / d_model)    # We compute it in log-space for numerical stability:    #   exp(2i * -log(10000) / d_model) = 1 / 10000^(2i/d_model)    div_term = torch.exp(        torch.arange(0, d_model, 2, dtype=torch.float) * (-np.log(10000.0) / d_model)    )  # shape: (d_model/2,)        pe[:, 0::2] = torch.sin(position * div_term)  # even dims: sin    pe[:, 1::2] = torch.cos(position * div_term)  # odd dims: cos        return ped_model = 64max_len = 128pe = sinusoidal_positional_encoding(max_len, d_model)print(f'PE shape: {pe.shape}  →  ({max_len} positions, {d_model} dimensions)')print(f'\nFirst 8 dims of PE[0]: {pe[0, :8].numpy().round(3)}')print(f'First 8 dims of PE[1]: {pe[1, :8].numpy().round(3)}')print(f'First 8 dims of PE[2]: {pe[2, :8].numpy().round(3)}')

## Part 3: Visualizing the Positional Fingerprints### 3a. The full PE matrix as a heatmapEach row is a position, each column is a dimension. Notice how:- Left columns (low `i`) oscillate rapidly → high frequency ("seconds hand")- Right columns (high `i`) change very slowly → low frequency ("year hand")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))im = ax.imshow(pe.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)ax.set_xlabel('Embedding Dimension')ax.set_ylabel('Position')ax.set_title('Sinusoidal Positional Encoding Matrix (128 positions × 64 dims)')plt.colorbar(im, ax=ax, label='Value')plt.tight_layout()plt.show()

### 3b. Individual dimension pairs — the "clock hands"Let's plot a few sin/cos pairs to see the different frequencies:

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)positions = np.arange(max_len)dim_pairs = [0, 4, 16, 31]  # i values → dimension pairs (2i, 2i+1)colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']for ax, i, color in zip(axes, dim_pairs, colors):    freq = 1.0 / (10000 ** (2 * i / d_model))    wavelength = 2 * np.pi / freq if freq > 0 else float('inf')        ax.plot(positions, pe[:, 2*i].numpy(), label=f'sin (dim {2*i})', color=color, linewidth=2)    ax.plot(positions, pe[:, 2*i+1].numpy(), label=f'cos (dim {2*i+1})', color=color,             linewidth=2, linestyle='--', alpha=0.7)    ax.set_ylabel(f'i={i}')    ax.set_title(f'Dimension pair i={i}  |  wavelength ≈ {wavelength:.1f} positions  |  freq = {freq:.6f}',                  fontsize=10, loc='left')    ax.legend(loc='upper right', fontsize=8)    ax.set_ylim(-1.3, 1.3)axes[-1].set_xlabel('Position in Sequence')fig.suptitle('Different Dimension Pairs = Different Frequencies (like clock hands)', fontsize=13, y=1.01)plt.tight_layout()plt.show()print("Notice:")print("  • i=0  → wavelength ≈ 6.3 (completes a full cycle every ~6 positions)")print("  • i=31 → wavelength ≈ 628318 (barely changes across 128 positions)")print("  • Together they create a UNIQUE pattern for each position")

### 3c. Each position has a unique fingerprintLet's verify: are the PE vectors for different positions actually distinct?We'll compute cosine similarity between all pairs of positions.

In [ ]:
# Cosine similarity between all position pairspe_normalized = pe / pe.norm(dim=1, keepdim=True)similarity = pe_normalized @ pe_normalized.Tfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Full similarity matrixim0 = axes[0].imshow(similarity.numpy(), cmap='viridis', aspect='auto')axes[0].set_xlabel('Position')axes[0].set_ylabel('Position')axes[0].set_title('Cosine Similarity Between All Position Pairs')plt.colorbar(im0, ax=axes[0])# Similarity of position 0 with all othersaxes[1].plot(similarity[0].numpy(), color='#e74c3c', linewidth=2, label='pos 0 vs all')axes[1].plot(similarity[32].numpy(), color='#3498db', linewidth=2, label='pos 32 vs all')axes[1].plot(similarity[64].numpy(), color='#2ecc71', linewidth=2, label='pos 64 vs all')axes[1].set_xlabel('Position')axes[1].set_ylabel('Cosine Similarity')axes[1].set_title('Similarity Decay from Reference Positions')axes[1].legend()plt.tight_layout()plt.show()print("Key observations:")print("  • Diagonal is 1.0 (each position is identical to itself)")print("  • Nearby positions are more similar (smooth gradient near diagonal)")print("  • Distant positions are less similar → natural distance signal")print("  • Each position has a UNIQUE fingerprint in the embedding space")

## Part 4: How Positional Encoding Flows into Q, K, VThis is the crucial part. In the original transformer:```input_to_attention = token_embedding + positional_encodingQ = (token_emb + PE) @ W_qK = (token_emb + PE) @ W_k  V = (token_emb + PE) @ W_v```The addition means Q, K, V each contain **both** semantic and positional information.Let's expand the attention score to see what's really happening:$$q_m \cdot k_n = (e_m + p_m)W_q \cdot (e_n + p_n)W_k$$Expanding this dot product:$$= \underbrace{e_m W_q \cdot e_n W_k}_{\text{content-content}} + \underbrace{e_m W_q \cdot p_n W_k}_{\text{content-position}} + \underbrace{p_m W_q \cdot e_n W_k}_{\text{position-content}} + \underbrace{p_m W_q \cdot p_n W_k}_{\text{position-position}}$$Four terms! The model learns to use ALL of these:1. **Content↔Content**: "does this word relate to that word?" (pure semantics)2. **Content↔Position**: "does this word care about what's at position n?"3. **Position↔Content**: "does position m care about that word?"4. **Position↔Position**: "do these two positions inherently relate?" (pure positional bias)Let's visualize each term:

In [ ]:
torch.manual_seed(42)d_model = 32seq_len = 20# Simulate token embeddings (content) and positional encodingstoken_emb = torch.randn(seq_len, d_model) * 0.5  # random "word meanings"pos_enc = sinusoidal_positional_encoding(seq_len, d_model)# Learned projection matricesW_q = torch.randn(d_model, d_model) * 0.1W_k = torch.randn(d_model, d_model) * 0.1# Compute the four attention score components# Full: (e + p) W_q @ ((e + p) W_k)^Te_Wq = token_emb @ W_q  # content queriesp_Wq = pos_enc @ W_q    # position queriese_Wk = token_emb @ W_k  # content keysp_Wk = pos_enc @ W_k    # position keyscontent_content = e_Wq @ e_Wk.T    # term 1content_position = e_Wq @ p_Wk.T   # term 2position_content = p_Wq @ e_Wk.T   # term 3position_position = p_Wq @ p_Wk.T  # term 4full_score = content_content + content_position + position_content + position_positionfig, axes = plt.subplots(1, 5, figsize=(22, 4))titles = ['Content↔Content', 'Content↔Position', 'Position↔Content',           'Position↔Position', 'Full Score (sum)']matrices = [content_content, content_position, position_content,             position_position, full_score]for ax, title, mat in zip(axes, titles, matrices):    vmax = mat.abs().max().item()    im = ax.imshow(mat.detach().numpy(), cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)    ax.set_title(title, fontsize=10)    ax.set_xlabel('Key position')    ax.set_ylabel('Query position')    plt.colorbar(im, ax=ax, shrink=0.8)plt.suptitle('Decomposition of Attention Scores: Content vs Position Components', fontsize=13, y=1.05)plt.tight_layout()plt.show()print("Notice the Position↔Position term:")print("  • It has a clear diagonal structure — nearby positions attend to each other")print("  • This is INDEPENDENT of content — it's a pure positional bias")print("  • The model uses this to learn patterns like 'the next word' or 'the previous word'")

## Part 5: Attention With vs Without Positional EncodingLet's see how adding PE changes the attention pattern on a real-ish example:

In [ ]:
torch.manual_seed(123)d_model = 32sentence = ["The", "cat", "sat", "on", "the", "mat", "and", "the", "dog", "slept"]seq_len = len(sentence)# Simulate token embeddings (same word = same embedding)vocab_emb = {w: torch.randn(d_model) for w in set(sentence)}token_emb = torch.stack([vocab_emb[w] for w in sentence])  # (seq_len, d_model)pos_enc = sinusoidal_positional_encoding(seq_len, d_model)W_q = torch.randn(d_model, d_model) * 0.15W_k = torch.randn(d_model, d_model) * 0.15def compute_attention_weights(X):    Q = X @ W_q    K = X @ W_k    scores = Q @ K.T / (d_model ** 0.5)    return torch.softmax(scores, dim=-1)attn_no_pe = compute_attention_weights(token_emb)attn_with_pe = compute_attention_weights(token_emb + pos_enc)fig, axes = plt.subplots(1, 2, figsize=(14, 5))for ax, attn, title in zip(axes, [attn_no_pe, attn_with_pe],                              ['WITHOUT Positional Encoding', 'WITH Positional Encoding']):    im = ax.imshow(attn.detach().numpy(), cmap='Blues', aspect='auto', vmin=0)    ax.set_xticks(range(seq_len))    ax.set_yticks(range(seq_len))    ax.set_xticklabels(sentence, rotation=45, ha='right', fontsize=9)    ax.set_yticklabels(sentence, fontsize=9)    ax.set_title(title, fontsize=12)    ax.set_xlabel('Key (attending to)')    ax.set_ylabel('Query (from)')    plt.colorbar(im, ax=ax, shrink=0.8)plt.tight_layout()plt.show()print('WITHOUT PE:')print('  • "the" at positions 0, 4, 7 all have IDENTICAL attention patterns')print('  • The model cannot distinguish the first "the" from the last "the"')print()print('WITH PE:')print('  • Each "the" now has a DIFFERENT attention pattern based on its position')print('  • The model can learn "the word before a noun" vs "the word starting a clause"')

## Part 6: Why Sinusoidal PE Can't Scale Beyond Training LengthThe sinusoidal functions themselves extrapolate perfectly — sin and cos are defined for all positions.So why can't the model handle longer sequences?The problem is that **the model's learned weights (W_q, W_k) have only seen position interactionswithin the training range**. The Position↔Position attention patterns for distant positionsare out-of-distribution.Let's visualize this:

In [ ]:
# Train on 64 positions, try to use 256train_len = 64test_len = 256d_model = 64pe_train = sinusoidal_positional_encoding(train_len, d_model)pe_test = sinusoidal_positional_encoding(test_len, d_model)# The PE vectors themselves are fine — let's check their normsnorms_test = pe_test.norm(dim=1)fig, axes = plt.subplots(2, 2, figsize=(14, 10))# Plot 1: PE norms across positionsaxes[0, 0].plot(norms_test.numpy(), color='#3498db', linewidth=2)axes[0, 0].axvline(x=train_len, color='red', linestyle='--', label=f'Training boundary (pos {train_len})')axes[0, 0].set_xlabel('Position')axes[0, 0].set_ylabel('L2 Norm of PE vector')axes[0, 0].set_title('PE Vector Norms (stable — not the problem)')axes[0, 0].legend()# Plot 2: Cosine similarity — train positions vs ALL positionspe_test_norm = pe_test / pe_test.norm(dim=1, keepdim=True)sim_full = pe_test_norm @ pe_test_norm.Taxes[0, 1].imshow(sim_full.numpy(), cmap='viridis', aspect='auto')axes[0, 1].axhline(y=train_len, color='red', linestyle='--', linewidth=2)axes[0, 1].axvline(x=train_len, color='red', linestyle='--', linewidth=2)axes[0, 1].set_title('Cosine Similarity (full 256 positions)')axes[0, 1].set_xlabel('Position')axes[0, 1].set_ylabel('Position')# Plot 3: The REAL problem — dot product patterns change# Simulate learned W_q, W_k that were trained on 64 positionstorch.manual_seed(42)W_q = torch.randn(d_model, d_model) * 0.1W_k = torch.randn(d_model, d_model) * 0.1# Position-position attention scorespp_train = (pe_train @ W_q) @ (pe_train @ W_k).Tpp_test = (pe_test @ W_q) @ (pe_test @ W_k).Tim2 = axes[1, 0].imshow(pp_train.detach().numpy(), cmap='RdBu_r', aspect='auto')axes[1, 0].set_title(f'Position↔Position scores (training: 0-{train_len-1})')axes[1, 0].set_xlabel('Key position')axes[1, 0].set_ylabel('Query position')plt.colorbar(im2, ax=axes[1, 0], shrink=0.8)im3 = axes[1, 1].imshow(pp_test.detach().numpy(), cmap='RdBu_r', aspect='auto',                          vmin=pp_train.min().item(), vmax=pp_train.max().item())axes[1, 1].axhline(y=train_len, color='lime', linestyle='--', linewidth=2, label='Training boundary')axes[1, 1].axvline(x=train_len, color='lime', linestyle='--', linewidth=2)axes[1, 1].set_title(f'Position↔Position scores (extrapolated: 0-{test_len-1})')axes[1, 1].set_xlabel('Key position')axes[1, 1].set_ylabel('Query position')axes[1, 1].legend(loc='lower right')plt.colorbar(im3, ax=axes[1, 1], shrink=0.8)plt.suptitle('Why Sinusoidal PE Fails at Extrapolation', fontsize=14, y=1.02)plt.tight_layout()plt.show()print("The bottom-right quadrant of the extrapolated matrix (beyond green lines)")print("shows position↔position interactions the model has NEVER seen during training.")print()print("The learned W_q and W_k weights have no idea how to handle these patterns.")print("This is why the model's output degrades — it's not the PE that breaks,")print("it's the LEARNED ATTENTION PATTERNS that don't generalize.")

## Part 7: The Relative Position Property (and its limitation)The original paper claimed that sinusoidal PE allows the model to learn relative positionsbecause $PE(pos+k)$ can be expressed as a **linear transformation** of $PE(pos)$.Specifically, for any fixed offset $k$, there exists a matrix $M_k$ such that:$$PE(pos + k) = M_k \cdot PE(pos)$$This is because sin/cos addition formulas give us:$$\sin(pos \cdot \omega + k \cdot \omega) = \sin(pos \cdot \omega)\cos(k \cdot \omega) + \cos(pos \cdot \omega)\sin(k \cdot \omega)$$Let's verify this property:

In [ ]:
d_model = 16  # small for claritymax_len = 50pe = sinusoidal_positional_encoding(max_len, d_model)# For offset k, the linear transformation M_k is a block-diagonal matrix# of 2x2 rotation matricesdef build_relative_transform(k, d_model):    """Build the matrix M_k such that PE(pos+k) = M_k @ PE(pos)."""    M = torch.zeros(d_model, d_model)    for i in range(d_model // 2):        omega = 1.0 / (10000 ** (2 * i / d_model))        angle = k * omega        # 2x2 rotation block for dimension pair (2i, 2i+1)        M[2*i, 2*i] = np.cos(angle)      # sin(pos+k) depends on sin(pos)        M[2*i, 2*i+1] = np.sin(angle)    # sin(pos+k) depends on cos(pos)        M[2*i+1, 2*i] = -np.sin(angle)   # cos(pos+k) depends on sin(pos)        M[2*i+1, 2*i+1] = np.cos(angle)  # cos(pos+k) depends on cos(pos)    return M# Verify: PE(pos+k) should equal M_k @ PE(pos)k = 5M_k = build_relative_transform(k, d_model)pos = 10pe_pos = pe[pos]pe_pos_plus_k = pe[pos + k]pe_predicted = M_k @ pe_posprint(f"PE({pos+k}) actual:    {pe_pos_plus_k[:8].numpy().round(4)}")print(f"M_{k} @ PE({pos}):     {pe_predicted[:8].numpy().round(4)}")print(f"Max error:            {(pe_pos_plus_k - pe_predicted).abs().max().item():.2e}")print()# Visualize M_k for different offsetsfig, axes = plt.subplots(1, 4, figsize=(16, 4))for ax, k in zip(axes, [1, 3, 10, 25]):    M = build_relative_transform(k, d_model)    ax.imshow(M.numpy(), cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')    ax.set_title(f'M (offset k={k})', fontsize=10)    ax.set_xlabel('From dim')    ax.set_ylabel('To dim')plt.suptitle('Relative Position Transform Matrices — Block-Diagonal Rotations', fontsize=12, y=1.02)plt.tight_layout()plt.show()print("Each M_k is a block-diagonal matrix of 2×2 rotation matrices.")print("This means PE(pos+k) is a ROTATION of PE(pos) — the relationship is linear!")print()print("HOWEVER: the model must LEARN to exploit this linear relationship through W_q and W_k.")print("In practice, the model partially learns this, but not perfectly — which is why")print("RoPE (which DIRECTLY applies rotations to Q and K) works so much better.")

## Summary| Aspect | Sinusoidal PE ||--------|--------------|| **Type** | Absolute (added to embeddings) || **Uniqueness** | ✅ Each position gets a unique fingerprint || **Relative position** | ⚠️ Theoretically possible via linear transform, but model must learn it || **Extrapolation** | ❌ Model's learned W_q, W_k don't generalize beyond training length || **Where applied** | Added to input embeddings → flows into Q, K, AND V || **Scalability** | Limited to training sequence length |### The path forward:- **Learned PE** (BERT, GPT-2): Even worse — hard ceiling at max_position- **Relative PE** (Transformer-XL): Better, but complex and still limited- **RoPE**: Applies rotation directly to Q, K — relative position falls out naturally- **ALiBi**: Simplest — just a linear distance penalty on attention scoresNext notebook: **RoPE — how rotations in Q/K space solve the scaling problem** 🚀